# **Return of the Schema** for  a *General User Defined Knowledge Graph* 

This Notebook will guide you in using the provided KG-SaF-JDeX Workflow Functionalities to generate a new Schema and Data Dataset from your custom Knowledge Graph. Please follow the README guide before going forward. This guide provides an example dataset in the INPUT folder to test the functionalities. All the produces files will be created in the OUTPUT folder.  

This notebook expectes the following inputs:

- Any Knowledge Graph with both Schema and Data in a unique file (any format supported by the ROBOT Utility, the file will be converted to an intermediate OWL File)

And applies the following procedures following the KG-SaF-JDeX workflow:

#### Dataset Creation Steps

- Materialization and Realization
- Filtering of ABox Individuals and Object Properties
- Object Propety Assertion splitting using Coverage Criterion in Training, Test and Validatin Split
- Inversion Leakage check and filtering
- Class Assertions subset computation
- Schema Modularization based on ABox Signature
- Schema Decomposition in TBox and RBox (with subsequent division in Schema and Taxonomy)
- Fulle cleaned Ontology and Knowledge Graph Reconstruction


#### Dataset Machine Learning Post Processing Steps

- Conversion and serialization of object property assertion to TSV format
- Conversion and serialization of schema axioms to JSON format

In [73]:
from rdflib import Graph, RDF, OWL, BNode
from rdflib.term import URIRef
from pathlib import Path
import sys
import subprocess
import json
sys.path.append(str(Path.cwd().parent))
from kgsaf_jdex.utils.conventions.builtins import BUILTIN_URIS
from kgsaf_jdex.utils.modularization import SignatureModularizer, SchemaDecomposer 
from pykeen.triples import TriplesFactory
from pykeen.triples.splitting import CoverageSplitter
from kgsaf_jdex.utils.conversion import OWLConverter, TSVConverter
from pykeen.triples.leakage import unleak


def serialize(graph, path, robot_jar):
    xml_path = path.with_suffix(".xml")
    owl_path = path.with_suffix(".owl")
    graph.serialize(xml_path, format="xml")

    cmd = [
        "java",
        "-Xmx20G",
        "-jar", str(robot_jar),
        "merge",
        "--input", str(xml_path),
        "--output", str(owl_path)
    ]
    

    result = subprocess.run(cmd, capture_output=False, text=True)

    if result.returncode != 0:
        raise RuntimeError(f"ROBOT merge failed with return code {result.returncode}")

    xml_path.unlink()
    print(f"Serialized graph saved to {owl_path}")

## Base Parameters 

In [27]:
# Do you want to run reasoning services on your dataset? 
REASONER = True

# INPUT Graph file Location
KG_FILE = Path().cwd().absolute() / "input" / "kg_consistent.owl"

# OUTPUT Dataset Location
OUTPUT_PATH = Path().cwd().absolute() / "output"

# Name of the GRAPH (Will be the datset folder NAME)
DATASET_NAME = "kg_consistent"
DATASET_NAME = DATASET_NAME + "_reasoned" if REASONER else DATASET_NAME + "_base"
DATASET_PATH = OUTPUT_PATH / f"{DATASET_NAME}"

# Location of the ROBOT Jar
ROBOT_JAR = "/home/navis/robot/robot.jar"


print(f"Using resoner? \t\t{REASONER}")
print(f"Input Graph Location \t{KG_FILE}")
print(f"Output Dataset Folder \t{OUTPUT_PATH}")
print(f"Dataset Name \t\t{DATASET_NAME}")
print(f"Robot JAR \t\t{ROBOT_JAR}")
print(f"Output Dataset Path \t{DATASET_PATH}")


Using resoner? 		True
Input Graph Location 	/home/navis/dev/kg-saf/tutorial/input/kg_consistent.owl
Output Dataset Folder 	/home/navis/dev/kg-saf/tutorial/output
Dataset Name 		kg_consistent_reasoned
Robot JAR 		/home/navis/robot/robot.jar
Output Dataset Path 	/home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned


In [28]:
print(f"Creating base datset Folders...")

(DATASET_PATH / "abox" / "splits").mkdir(parents=True, exist_ok=True)
(DATASET_PATH / "mappings").mkdir(parents=True, exist_ok=True)
(DATASET_PATH / "tbox").mkdir(parents=True, exist_ok=True)
(DATASET_PATH / "rbox").mkdir(parents=True, exist_ok=True)

print(f"Done!")

Creating base datset Folders...
Done!


## (OPTIONAL) Reasoning Utilities

This block will run reasoning services for CONSISTENCY CHECK, MATERIALIZATION and REALIZATION

In [31]:
properties = [
    "SubClass",
    "EquivalentClass",
    "EquivalentObjectProperty",
    "InverseObjectProperties",
    "ObjectPropertyCharacteristic",
    "SubObjectProperty",
    "ObjectPropertyRange",
    "ObjectPropertyDomain",
    "ClassAssertion"
]

print("Axioms Generator:")
prop_string = ""
for p in properties:
    print(f"\t{p}")
    prop_string += " " + p


debug_path = DATASET_PATH / "debug.owl"
output_file = DATASET_PATH / "intermediate_kg.owl"


if REASONER:
    print("Running Reasoner on Target Ontology: Realization and Materialization")
    cmd = [
        "java",
        "-Xmx20G",
        "-jar", str(ROBOT_JAR),
        "reason",
        "-vvv",
        "--reasoner", "HermiT",
        "--input", str(KG_FILE),
        "--output", str(output_file),
        "--axiom-generators", prop_string,
        "--remove-redundant-subclass-axioms", "false",
        "--exclude-tautologies", "structural",
        "--include-indirect", "true",
        "-D", str(debug_path)
]
else:
    print("Running Reasoner on Target Ontology: Conversion to OWL Format")
    cmd = [
        "java", "-Xmx20G", "-jar", str(ROBOT_JAR),
        "merge",
        "-vvv",
        "--input", str(KG_FILE),
        "--output", str(output_file)
    ]

result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode == 0:
    print("Reasoning completed successfully!")
else:
    print("Reasoning failed with return code:", result.returncode)

Axioms Generator:
	SubClass
	EquivalentClass
	EquivalentObjectProperty
	InverseObjectProperties
	ObjectPropertyCharacteristic
	SubObjectProperty
	ObjectPropertyRange
	ObjectPropertyDomain
	ClassAssertion
Running Reasoner on Target Ontology: Realization and Materialization
2026-01-21 14:13:40,340 DEBUG org.obolibrary.robot.IOHelper - Loading ontology /home/navis/dev/kg-saf/tutorial/input/kg_consistent.owl with catalog file null
2026-01-21 14:13:40,342 DEBUG org.semanticweb.owlapi.utilities.Injector - Loading file META-INF/services/org.semanticweb.owlapi.model.OWLOntologyManager
2026-01-21 14:13:40,343 DEBUG org.semanticweb.owlapi.utilities.Injector - Loading URL for service jar:file:/home/navis/robot/robot.jar!/META-INF/services/org.semanticweb.owlapi.model.OWLOntologyManager
2026-01-21 14:13:40,343 DEBUG org.semanticweb.owlapi.utilities.Injector - Loading URL for service jar:file:/home/navis/robot/robot.jar!/META-INF/services/org.semanticweb.owlapi.model.OWLOntologyManager
2026-01-21 1

# ABOX Filtering: NamedIndividuals and ObjectProperties

Removal of NamedIndividuals also defined as Classes and ObjectProperties also defined as DatatypeProperties

In [33]:
print("Parsing and Loading Target Knowledge Graph...")
kg = Graph()
kg.parse(output_file)
print("Done!")

Parsing and Loading Target Knowledge Graph...
Done!


#### Triples

In [34]:
print("Filtering <NamedIndividual, ObjectPropety, NamedIndividual> Triples")

triples_graph = Graph()
for s,p,o in kg:
    if (s, RDF.type, OWL.NamedIndividual) in kg and (p, RDF.type, OWL.ObjectProperty) in kg and (o, RDF.type, OWL.NamedIndividual) in kg:
        triples_graph.add((s,p,o))

print(f"Initial Dataset: {len(triples_graph)} triples found (OWL.NamedIndividual, OWL.ObjectProperty, OWL.NamedIndividual) Found")

Filtering <NamedIndividual, ObjectPropety, NamedIndividual> Triples
Initial Dataset: 76943 triples found (OWL.NamedIndividual, OWL.ObjectProperty, OWL.NamedIndividual) Found


#### Predicates

In [36]:
predicates = set()

print("Analyzing Object Properties")

for prop in triples_graph.predicates(None, None):
    if (prop, RDF.type, OWL.DatatypeProperty) not in kg:
        predicates.add(prop)
    else:
        print(f"Predicate {prop} removed from dataset. Defined as DatatypeProperty")

print(f"Initial Dataset: {len(predicates)} predicates (OWL.ObjectProperty) Found")


Analyzing Object Properties
Initial Dataset: 25 predicates (OWL.ObjectProperty) Found


#### Individuals

In [37]:
individuals = set()

print("Analyzing SUBJECTS")
for ind in triples_graph.subjects(None, None):
    if (ind, RDF.type, OWL.Class) not in kg:
        individuals.add(ind)
    else:
        print(f"Individual {ind} removed from dataset. Defined as Class")

print("Analyzing OBJECTS")
for ind in triples_graph.objects(None, None):
    if (ind, RDF.type, OWL.Class) not in kg:
        individuals.add(ind)
    else:
        print(f"Individual {ind} removed from dataset. Defined as Class")

print(f"Dataset will contain a total of {len(individuals)} individuals (OWL.NamedIndividual)")

Analyzing SUBJECTS
Analyzing OBJECTS
Dataset will contain a total of 29767 individuals (OWL.NamedIndividual)


#### Filtering

In [ ]:
print(f"Removing Triples...")

for s,p,o in triples_graph:
    if (s not in individuals) or (o not in individuals):
        triples_graph.remove((s,p,o))

print(f"Dataset will contain a total of {len(triples_graph)} triples (OWL.NamedIndividual, OWL.ObjectProperty, OWL.NamedIndividual)")

print("Serializing Intermediate File...")
triples_graph.serialize(DATASET_PATH / "obj_prop_assertion.nt", format="nt", robot_jar=ROBOT_JAR)

with open(DATASET_PATH / "obj_prop_assertion.tsv", "w") as f:
    for s, p, o in triples_graph:
        f.write(f"{s}\t{p}\t{o}\n")
print("Done")

Removing Triples...
Dataset will contain a total of 76943 triples (OWL.NamedIndividual, OWL.ObjectProperty, OWL.NamedIndividual)
Serializing Intermediate File...


/home/navis/.pyenv/versions/oml/lib/python3.12/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


Done


## Dataset Splitting: Covarage Splitting in Train Validation and Test, Inversion Leakage Filtering

Modify this cell as need following PyKEEN guidelines to change the Splitting Criterion. Reporting from PyKEEN Docs:

```text
PyKEEN currently only supports the creation of transductive splits. 
One reason for this is that in the inductive setting, there are several different ways to create inductive splits and the literature uses different ones, cf. https://arxiv.org/abs/2107.04894. 
You can still use PyKEEN for inductive link prediction with existing data sets, or with new inductive data sets that you create. 
For a general discussion of inductive link prediction, see Inductive Link Prediction.```

In [51]:
triples = TriplesFactory.from_path(DATASET_PATH / "obj_prop_assertion.tsv")

entity_mappings = {v:k for k,v in triples.entity_id_to_label.items()}
relation_mappings = {v:k for k,v in triples.relation_id_to_label.items()}

train, valid, test = triples.split(
    ratios=[0.85, 0.05, 0.1],
    random_state=42,
    method=CoverageSplitter(),      
)

train_clean = TriplesFactory.from_labeled_triples(
    triples=train.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

valid_clean = TriplesFactory.from_labeled_triples(
    triples=valid.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

test_clean = TriplesFactory.from_labeled_triples(
    triples=test.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

train_unleak, valid_unleak, test_unleak = unleak(
    train_clean,
    *[valid_clean, test_clean],
    n=None,
    minimum_frequency=0.97
    )


Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.


In [ ]:
print("Serializing NT Splits and Full Dataset...")

targets = [
    (DATASET_PATH / "abox/splits/train", train_unleak.triples),
    (DATASET_PATH / "abox/splits/valid", valid_unleak.triples),
    (DATASET_PATH / "abox/splits/test", test_unleak.triples)
]

for path, split in targets:
    out_graph = Graph()
    for triple in split:
        s = URIRef(triple[0])
        p = URIRef(triple[1])
        o = URIRef(triple[2])
        out_graph.add((URIRef(s), URIRef(p), URIRef(o)))

    print(f"\tSerializing {path}")
    out_graph.serialize(path.with_suffix(".nt"), format="nt")

print(f"\tSerializing  Full Dataset")
!cat {DATASET_PATH}/abox/splits/*.nt > {DATASET_PATH}/abox/obj_prop_assertions.nt
print("Done")

Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.


Serializing NT Splits and Full Dataset...
	Serializing /home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned/abox/splits/train


/home/navis/.pyenv/versions/oml/lib/python3.12/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


	Serializing /home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned/abox/splits/valid
	Serializing /home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned/abox/splits/test
	Serializing  Full Dataset
Done


## ABox Serialization: NamedIndividuals and ClassAssertions

In [57]:
print("Serializing NamedIndividuals...")
out_graph = Graph()

for ind in individuals:
    out_graph.add((ind, RDF.type, OWL.NamedIndividual))

serialize(out_graph, DATASET_PATH / "abox" / "individuals", robot_jar=ROBOT_JAR)
del out_graph
print("Done!")

Serializing NamedIndividuals...
Serialized graph saved to /home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned/abox/individuals.owl
Done!


In [58]:
print("Serializing ClassAssertions...")
class_assertions_graph = Graph()

for ind in individuals:
    for ca in set(kg.objects(ind, RDF.type)) - BUILTIN_URIS:
        if (ca, RDF.type, OWL.Class) in kg:
            class_assertions_graph.add((ind, RDF.type, ca))
        else:
            print(f"Not a Class {ca}")

serialize(class_assertions_graph, DATASET_PATH / "abox" / "class_assertions", robot_jar=ROBOT_JAR)
print("Done!")

Serializing ClassAssertions...
Serialized graph saved to /home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned/abox/class_assertions.owl
Done!


# Schema Subset: Signature Based Modularization

In [62]:
seed_obj_props = predicates
print("Seed Object Properties", len(seed_obj_props))

seed_classes = set(class_assertions_graph.objects(None, RDF.type))
print("Seed Classes", len(seed_classes))

modularizer = SignatureModularizer(kg, seed_classes | seed_obj_props)
out_graph = modularizer.modularize(verbose=False)

serialize(out_graph, DATASET_PATH / "ontology", robot_jar=ROBOT_JAR)

Seed Object Properties 25
Seed Classes 65
Serialized graph saved to /home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned/ontology.owl


## Schema Decomposition: TBox and RBox

In [64]:
onto_graph = Graph()
onto_graph.parse(DATASET_PATH / "ontology.owl")

decomposer = SchemaDecomposer(onto_graph)
rbox_graph, taxonomy_graph, schema_graph = decomposer.decompose(verbose=False)

serialize(rbox_graph, DATASET_PATH / "rbox" / "roles", robot_jar=ROBOT_JAR)
serialize(taxonomy_graph, DATASET_PATH / "tbox" / "taxonomy", robot_jar=ROBOT_JAR)
serialize(schema_graph, DATASET_PATH / "tbox" / "schema", robot_jar=ROBOT_JAR)


Serialized graph saved to /home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned/rbox/roles.owl
Serialized graph saved to /home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned/tbox/taxonomy.owl
Serialized graph saved to /home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned/tbox/schema.owl


# Ontology and Knowledge Graph Recomposition

In [65]:
(DATASET_PATH / "intermediate_kg.owl").unlink()
(DATASET_PATH / "obj_prop_assertion.nt").unlink()
(DATASET_PATH / "obj_prop_assertion.tsv").unlink()

In [66]:
import kgsaf_jdex.utils.conventions.paths as pcc

inputs = [
    DATASET_PATH / pcc.ONTOLOGY,
    DATASET_PATH / pcc.INDIVIDUALS,
    DATASET_PATH / pcc.RDF_TRIPLES,
    DATASET_PATH / pcc.RDF_CLASS_ASSERTIONS
]

output_file = DATASET_PATH / "knowledge_graph.owl"
cmd = ["java", "-Xmx20G", "-jar", str(ROBOT_JAR), "merge"]

for infile in inputs:
    cmd += ["--input", str(infile)]

cmd += ["--output", str(output_file)]

result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode != 0:
    raise RuntimeError(f"ROBOT merge failed with return code {result.returncode}")

print(f"Merged knowledge graph saved to {output_file}")

Merged knowledge graph saved to /home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned/knowledge_graph.owl


## Machine Learning and Torch Utilities: ID Mapping, TSV Conversion, JSON Conversion

In [67]:
print(f"\t Converting Dataset {DATASET_PATH.name}")
processor = TSVConverter(DATASET_PATH)
processor.convert()
processor.serialize()

	 Converting Dataset kg_consistent_reasoned


In [68]:
print(f"\tProcessing Dataset {DATASET_PATH.name}")
processor = OWLConverter(DATASET_PATH)
processor.preprocess(verbose=False)
processor.serialize()

	Processing Dataset kg_consistent_reasoned
Processing Dataset at /home/navis/dev/kg-saf/tutorial/output/kg_consistent_reasoned
Processing Taxonomy
Processing Class Assertions
Processing Object Property Hierarchy
Processing Object Property Domain and Range


In [74]:
onto = Graph()
onto.parse(DATASET_PATH / pcc.ONTOLOGY)

print("Loaded Ontology")

ind_onto = Graph()
ind_onto.parse(DATASET_PATH / pcc.INDIVIDUALS)

print("Loaded Individuals")

classes =  set(onto.subjects(RDF.type, OWL.Class)) - BUILTIN_URIS
classes = {c for c in classes if not isinstance(c, BNode)}

properties = set(onto.subjects(RDF.type, OWL.ObjectProperty)) - BUILTIN_URIS
individuals = set(ind_onto.subjects(RDF.type, OWL.NamedIndividual)) - BUILTIN_URIS

classes = list(classes)
properties = list(properties)
individuals = list(individuals)

classes.sort()
properties.sort()
individuals.sort()

print("Classes", len(classes))
print("Properties", len(properties))
print("Individuals", len(individuals))



classes_mappings = {str(c):i for i,c in enumerate(classes)}
individuals_mappings = {str(c):i for i,c in enumerate(individuals)}
properties_mappings = {str(c):i for i,c in enumerate(properties)}

(DATASET_PATH / pcc.MAPPINGS).mkdir(exist_ok=True, parents=True)

with open(DATASET_PATH / pcc.CLASS_MAPPINGS, "w") as f:
    json.dump(classes_mappings, f, indent=4)

with open(DATASET_PATH / pcc.INDIVIDUAL_MAPPINGS, "w") as f:
    json.dump(individuals_mappings, f, indent=4)

with open(DATASET_PATH / pcc.OBJ_PROP_MAPPINGS, "w") as f:
    json.dump(properties_mappings, f, indent=4)

Loaded Ontology
Loaded Individuals
Classes 94
Properties 71
Individuals 29767
